# 05c — Clustering GMM

3 versiones × 9 K = 27 modelos. BIC/AIC para seleccionar K.

In [1]:
import pandas as pd
import numpy as np
import time, joblib
from pathlib import Path
from datetime import datetime
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_QC = ROOT / 'data' / 'data_qc_gustos'
DATA_MODELS = ROOT / 'data' / 'data_models_gustos'
INFORMES_BASE = ROOT / 'informes' / 'fase4_gustos' / '04_clustering'
DATA_MODELS.mkdir(parents=True, exist_ok=True)

MASTER_VERSIONS = {
    'v1_conservative': DATA_QC / 'master_table_gustos_v1_conservative.parquet',
    'v2_intermediate': DATA_QC / 'master_table_gustos_v2_intermediate.parquet',
    'v3_aggressive':   DATA_QC / 'master_table_gustos_v3_aggressive.parquet',
}

def reduce_high_card(df, max_unique=10):
    out = df.copy()
    for cat in out.select_dtypes(include=['object', 'category']).columns.tolist():
        n = out[cat].nunique(dropna=True)
        if n > max_unique:
            top = out[cat].value_counts().head(max_unique).index.tolist()
            out[cat] = out[cat].where(out[cat].isin(top), 'OTHER')
    return out

def preprocess_master(master_df, nan_threshold_zero=0.30):
    """Returns (X_scaled, preprocessor, feature_names_out)."""
    df = master_df.drop(columns=['user_id']).copy()
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
    nan_pcts = df[numeric_cols].isna().mean()
    num_low = nan_pcts[nan_pcts < nan_threshold_zero].index.tolist()
    num_high = nan_pcts[nan_pcts >= nan_threshold_zero].index.tolist()
    print(f'  num_low_nan: {len(num_low)} | num_high_nan: {len(num_high)} | cat: {len(categorical_cols)}')
    df = reduce_high_card(df, max_unique=10)
    preproc = ColumnTransformer([
        ('num_low', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', RobustScaler())]), num_low),
        ('num_high', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value=0)), ('sc', RobustScaler())]), num_high),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols),
    ])
    X = preproc.fit_transform(df)
    feature_names = preproc.get_feature_names_out()
    return X, preproc, feature_names

def load_and_preprocess(version_name):
    master = pd.read_parquet(MASTER_VERSIONS[version_name])
    print(f'[{version_name}] shape={master.shape}')
    user_ids = master['user_id'].values
    X, preproc, names = preprocess_master(master)
    return X, user_ids, preproc, names, master.shape

from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REPORT = INFORMES_BASE / '05c_gmm'
REPORT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
K_RANGE = list(range(2, 11))
SAMPLE_FOR_SILHOUETTE = 20000
SAMPLE_TRAIN_GMM = 60000  # GMM con N>60k es lento con full covariance
results = []


In [2]:

for version in MASTER_VERSIONS:
    print(f'\n=== {version} ===')
    X, user_ids, preproc, names, shape = load_and_preprocess(version)
    print(f'  X shape: {X.shape}')

    rng = np.random.RandomState(RANDOM_STATE)
    if len(X) > SAMPLE_TRAIN_GMM:
        train_idx = rng.choice(len(X), SAMPLE_TRAIN_GMM, replace=False)
        X_train = X[train_idx]
    else:
        X_train = X

    sil_idx = rng.choice(len(X), min(SAMPLE_FOR_SILHOUETTE, len(X)), replace=False)
    X_sil = X[sil_idx]

    bics = []; aics = []; sils = []
    for k in K_RANGE:
        t0 = time.time()
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=RANDOM_STATE, max_iter=200, n_init=1)
        gmm.fit(X_train)
        labels_all = gmm.predict(X)
        labels_sub = gmm.predict(X_sil)
        try:
            sil = float(silhouette_score(X_sil, labels_sub)) if len(set(labels_sub)) > 1 else float('nan')
        except Exception:
            sil = float('nan')
        try:
            db = float(davies_bouldin_score(X, labels_all))
        except Exception:
            db = float('nan')
        bic = float(gmm.bic(X_train))
        aic = float(gmm.aic(X_train))
        bics.append(bic); aics.append(aic); sils.append(sil)
        results.append({
            'algorithm': 'GMM', 'version': version, 'k': k,
            'silhouette': sil, 'davies_bouldin': db, 'bic': bic, 'aic': aic,
            'n_clusters_actual': int(len(set(labels_all))),
            'elapsed_s': time.time()-t0,
        })
        print(f'  K={k}: sil={sil:.4f} db={db:.4f} bic={bic:.0f} t={time.time()-t0:.1f}s')
        joblib.dump({'model': gmm, 'labels': labels_all, 'user_ids': user_ids}, DATA_MODELS / f'gmm_{version}_k{k}.joblib')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(K_RANGE, bics, marker='o', label='BIC'); axes[0].plot(K_RANGE, aics, marker='s', color='C1', label='AIC')
    axes[0].set_xlabel('K'); axes[0].set_ylabel('Information criterion'); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title(f'BIC/AIC — {version}')
    axes[1].plot(K_RANGE, sils, marker='o', color='C2'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].axhline(0.15, color='red', ls='--', alpha=0.4); axes[1].set_title(f'Silhouette — {version}'); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(REPORT / f'bic_aic_{version}.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'plot {version}')

res_df = pd.DataFrame(results)
res_df.to_csv(REPORT / 'metrics.csv', index=False)
best_bic = res_df.loc[res_df.groupby('version')['bic'].idxmin()][['version','k','silhouette','davies_bouldin','bic']]
best_sil = res_df.loc[res_df.groupby('version')['silhouette'].idxmax()][['version','k','silhouette','davies_bouldin','bic']]
print('\nBest BIC:'); print(best_bic.to_string(index=False))
print('\nBest silhouette:'); print(best_sil.to_string(index=False))

lines = [f'# 05c GMM', f'**Fecha**: {datetime.now().strftime("%Y-%m-%d %H:%M")}', '',
         '## K óptimo por BIC mínimo', '', '| Versión | K | silhouette | BIC |', '|---|---:|---:|---:|']
for _, r in best_bic.iterrows():
    lines.append(f"| {r['version']} | {int(r['k'])} | {r['silhouette']:.4f} | {r['bic']:.0f} |")
lines += ['', '## K óptimo por max silhouette', '', '| Versión | K | silhouette | BIC |', '|---|---:|---:|---:|']
for _, r in best_sil.iterrows():
    lines.append(f"| {r['version']} | {int(r['k'])} | {r['silhouette']:.4f} | {r['bic']:.0f} |")
(REPORT / 'execution_report.md').write_text('\n'.join(lines))



=== v1_conservative ===


[v1_conservative] shape=(114412, 115)
  num_low_nan: 86 | num_high_nan: 24 | cat: 4


/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_4065/2098438774.py:36: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
/var/folders/0f/3zq36dr1451_ttvt93n947ph0000gw/T/ipykernel_4065/2098438774.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence t

  X shape: (114412, 132)


  K=2: sil=0.9961 db=0.3582 bic=-4023581 t=2.5s


...

  K=7: sil=0.5910 db=0.7818 bic=-5346764 t=3.4s


  K=8: sil=0.6386 db=1.1173 bic=-11498484 t=6.0s


  K=9: sil=0.6386 db=1.0256 bic=-11427966 t=6.5s


  K=10: sil=0.6386 db=0.8839 bic=-11357181 t=7.1s
plot v3_aggressive

Best BIC:
        version  k  silhouette  davies_bouldin           bic
v1_conservative  8    0.623350        1.204137 -1.975500e+07
v2_intermediate  8    0.638596        1.123698 -1.390262e+07
  v3_aggressive  8    0.638600        1.117293 -1.149848e+07

Best silhouette:
        version  k  silhouette  davies_bouldin           bic
v1_conservative  2    0.996144        0.358151 -4.023581e+06
v2_intermediate  2    0.996144        0.358151  1.641028e+06
  v3_aggressive  2    0.996144        0.358151  2.515213e+06


472